# Media Misinformation — Master Notebook (Resumable)

**Project:** PTDATAX-267 multi-event Reddit corpus analysis · HICSS-60 misinformation-and-natural-disasters paper.

## ⚠ Read this first — 120-minute DataX sessions

This notebook is **resumable**. Every expensive step saves its output to disk the moment it finishes, and skips itself on re-run if its cache is present. If your DataX session times out mid-pipeline:

1. Start a fresh DataX session.
2. Open this notebook.
3. **Run All** (or step through from the top).
4. Cached steps print `✓ Resumed: …` and skip immediately. The first uncached step is where work resumes.

You may need 2–3 sessions to get all the way through the first time. After that, every re-run is fast because the caches stay.

**Checkpoints live in** `MediaMisinfoResults/checkpoints/` (named with `01_`, `02_`, … prefixes matching section numbers).

**To force a re-compute** of a specific stage, flip its flag in `FORCE` (§1.3). To start completely over, see the **Reset** cell in §9.

## What this notebook does

1. Ingests a multi-format corpus (`.jsonl`, `.json`, `.csv`, `.bib`, …) from a Corral directory, with streaming JSON support so multi-GB Reddit dumps don't OOM.
2. Partitions the corpus using the lead author's **keyword groups** and **boolean searches** (AND across required groups; OR within bracketed alternatives).
3. Discovers LDA sub-topics *within* each boolean partition, with metadata-aware stopwords.
4. Maps each sub-topic to a science backbone — either built from an ETO Map-of-Science CSV export, or the built-in inline scaffold.
5. Runs Kleinberg burst detection per event around day-zero timestamps.
6. Produces cross-filter comparisons (match counts, aligned timelines, Jaccard).
7. Exports parquet for the matched corpus + CSV aggregates + a Markdown run report.

## Provenance

Each code cell opens with a comment header:

```
# ═══ §X.Y | SOURCE: <origin> ═══
```

`SB` = `SemanticBridge_Burst_Integrated_Floods.ipynb` · `NB1`/`NB2`/`NB3` = the three MediaMisinfo cookbooks · `NEW` = consolidation glue · `WRAPPED` = original logic + checkpoint wrapper added here.


## 1. Setup

In [ ]:
# ═══ §1.1 | SOURCE: NB2 Cell 4 + SB Cell 3 (paths + event anchors) ═══
from pathlib import Path
import os

CANDIDATE_CORPUS_PATHS = [
    Path("/corral-repl/tacc/aci/PT2050/projects/PTDATAX-267/Data/For_HICCS"),
    Path("/corral-repl/tacc/aci/PT2050/projects/PTDATAX-267/Data"),
    Path(os.getcwd()) / "data",
]
CORPUS_PATH = next((p for p in CANDIDATE_CORPUS_PATHS if p.exists()),
                   CANDIDATE_CORPUS_PATHS[-1])

OUTPUT_DIR = Path(os.getcwd()) / "MediaMisinfoResults"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_DAY_ZERO = {
    "NC_Helene": "2024-09-26",
    "TX_Flood":  "2025-07-04",
    # add more here, e.g. "LA_Fires": "2025-01-07"
}
WINDOW_DAYS = 60

_kind = ("directory" if CORPUS_PATH.is_dir()
         else "file"  if CORPUS_PATH.is_file() else "missing")
print(f"Corpus source : {CORPUS_PATH}  ({_kind})")
print(f"Output dir    : {OUTPUT_DIR}")
print(f"Events        : {list(EVENT_DAY_ZERO.keys())}  ±{WINDOW_DAYS} days")


In [ ]:
# ═══ §1.2 | SOURCE: SB Cell 5 + NB2 _ensure pattern (dependencies) ═══
import sys, subprocess, importlib

def _ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _p, _m in [("plotly", "plotly"), ("networkx", "networkx"),
               ("scikit-learn", "sklearn"), ("bibtexparser", "bibtexparser"),
               ("pyarrow", "pyarrow"), ("ijson", "ijson"),
               ("python-docx", "docx"), ("pypdf", "pypdf"),
               ("tabulate", "tabulate")]:
    _ensure(_p, _m)

import json, csv, re, math, time
from datetime import datetime, timezone, timedelta
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go

import bibtexparser
from bibtexparser.bparser import BibTexParser
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from docx import Document as _Docx
from pypdf import PdfReader as _PdfReader

print(f"✓ Libraries ready (pandas {pd.__version__}, sklearn, plotly, networkx)")


In [ ]:
# ═══ §1.3 | NEW: Checkpoint configuration ═══════════════════════════════
# Every expensive step in this notebook saves its output to disk and loads
# from disk on re-run. If a session times out, re-open in a fresh session
# and Run All — cached steps skip automatically.

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# All checkpoint file paths in one place — never hardcoded elsewhere.
CKPT = {
    "raw_corpus":    CHECKPOINT_DIR / "01_df_raw.parquet",
    "group_scans":   CHECKPOINT_DIR / "02_group_scans.parquet",
    "matched_df":    CHECKPOINT_DIR / "03_df_matched.parquet",
    "match_summary": CHECKPOINT_DIR / "03_match_summary.csv",
    "subtopics":     CHECKPOINT_DIR / "04_subtopics.parquet",
    "llm_labels":    CHECKPOINT_DIR / "04_llm_labels.csv",
    "burst_data":    CHECKPOINT_DIR / "06_burst_data",   # directory: one file per event
}
CKPT["burst_data"].mkdir(parents=True, exist_ok=True)

# Force-recompute flags — flip any to True to invalidate that cache and
# recompute on the next run. Defaults (all False) use cache when present.
FORCE = {
    "raw_corpus":  False,
    "group_scans": False,
    "matched_df":  False,
    "subtopics":   False,
    "llm_labels":  False,
    "bursts":      False,
}

# Helper used throughout for displaying file/dir sizes
def _fmt_size(p):
    if not p.exists(): return "  --"
    s = p.stat().st_size if p.is_file() else sum(
        f.stat().st_size for f in p.rglob("*") if f.is_file())
    for unit in ("B","KB","MB","GB"):
        if s < 1024: return f"{s:6.1f} {unit}"
        s /= 1024
    return f"{s:6.1f} TB"

print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"  {sum(1 for k, p in CKPT.items() if p.exists())} of {len(CKPT)} caches present")


In [ ]:
# ═══ §1.4 | NEW: Resume diagnostic — "where am I?" ═══════════════════
# Prints which stages are cached (✓), which will be force-recomputed, and
# which still need work. Run this any time to check pipeline status.

_STAGES = [
    ("§2.2  corpus load",       "raw_corpus"),
    ("§3.2a group scans",       "group_scans"),
    ("§3.2b matched df",        "matched_df"),
    ("§4.2  sub-topics (LDA)",  "subtopics"),
    ("§4.3  LLM labels",        "llm_labels"),
    ("§6.2  burst data",        "burst_data"),
]

print(f"{'Stage':<25} {'Status':<14} {'Size':<10} {'Last modified':<19}")
print("-" * 70)
for label, key in _STAGES:
    p = CKPT[key]
    if p.exists():
        forced = FORCE.get(key, False)
        status = "FORCE-RERUN" if forced else "✓ cached"
        mt_src = p.stat().st_mtime if p.is_file() else max(
            (f.stat().st_mtime for f in p.rglob("*") if f.is_file()), default=0)
        mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(mt_src)) if mt_src else ""
    else:
        status, mtime = "  pending", ""
    print(f"{label:<25} {status:<14} {_fmt_size(p):<10} {mtime:<19}")

# Estimate remaining work
_pending = [label for label, key in _STAGES
            if not CKPT[key].exists() or FORCE.get(key, False)]
if _pending:
    print(f"\n→ {len(_pending)} stage(s) to compute this session: " + ", ".join(_pending))
else:
    print("\n✓ All stages cached — re-run will be near-instant (~1 minute).")


## 2. Multi-format corpus ingest

Loads `CORPUS_PATH` (single file or recursive directory walk) into `df_raw`. Reddit metadata (`timestamp`, `subreddit`, `permalink`, `author`) is captured as first-class columns. Large `.json` files stream-parse via `ijson` so multi-GB Reddit dumps don't OOM.

In [ ]:
# ═══ §2.1 | SOURCE: NB2 Cell 13 (multi-format loader, ijson streaming) ═══
# Extended to extract Reddit metadata for §6 burst analysis.
# Pure function definitions — no work happens here.

CORPUS_GLOB_PATTERNS = ["*.bib", "*.txt", "*.md", "*.json", "*.jsonl",
                        "*.ndjson", "*.csv", "*.tsv", "*.docx", "*.pdf"]
CORPUS_MAX_FILES = 5000
CORPUS_MIN_CHARS = 50
CANON_COLS = ["id", "type", "year", "title", "abstract", "keywords",
              "journal", "author", "doi", "subreddit", "permalink",
              "timestamp", "source_file", "text_content"]


def _clean(text):
    if text is None: return ""
    t = re.sub(r"[{}\\]", " ", str(text))
    return re.sub(r"\s+", " ", t).strip()


def _to_dt(ts):
    if ts is None or ts == "": return None
    try:
        if isinstance(ts, (int, float)):
            return datetime.fromtimestamp(int(ts), tz=timezone.utc)
        s = str(ts)
        if s.isdigit() and len(s) in (9, 10, 11):
            return datetime.fromtimestamp(int(s), tz=timezone.utc)
        return datetime.fromisoformat(s.replace("Z", "+00:00"))
    except Exception:
        return None


def _year_from_anything(*candidates):
    for c in candidates:
        if c is None or c == "": continue
        s = str(c)
        if re.fullmatch(r"\d{9,11}", s):
            try: return datetime.fromtimestamp(int(s), tz=timezone.utc).year
            except Exception: pass
        m = re.search(r"(19|20)\d{2}", s)
        if m:
            y = int(m.group(0))
            if 1900 <= y <= datetime.now().year + 1: return y
    return None


def _record_from_dict(d, source_file):
    g = lambda *keys: next((d[k] for k in keys
                            if k in d and d[k] not in (None, "")), "")
    ts = _to_dt(d.get("created_utc") or d.get("created") or d.get("date")
                or d.get("published") or d.get("timestamp"))
    return {
        "id":          _clean(g("id", "ID", "doi", "DOI")) or source_file,
        "type":        _clean(g("type", "ENTRYTYPE", "kind")) or "doc",
        "year":        ts.year if ts else _year_from_anything(d.get("year")),
        "title":       _clean(g("title", "Title")),
        "abstract":    _clean(g("abstract", "Abstract", "selftext", "body",
                                "text", "content", "summary")),
        "keywords":    _clean(g("keywords", "tags")),
        "journal":     _clean(g("journal", "venue", "booktitle", "source")),
        "author":      _clean(g("author", "authors", "by")),
        "doi":         _clean(g("doi", "DOI")),
        "subreddit":   _clean(g("subreddit")),
        "permalink":   _clean(g("permalink")),
        "timestamp":   ts,
        "source_file": source_file,
    }


def _record_from_text(text, source_file, title=None):
    return {
        "id": source_file, "type": "doc",
        "year": _year_from_anything(source_file, text[:500]),
        "title": _clean(title) or Path(source_file).stem,
        "abstract": _clean(text),
        "keywords": "", "journal": "", "author": "", "doi": "",
        "subreddit": "", "permalink": "", "timestamp": None,
        "source_file": source_file,
    }


def _load_bib(path):
    parser = BibTexParser(common_strings=True)
    parser.ignore_nonstandard_types = False
    parser.homogenize_fields = True
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        db = bibtexparser.load(fh, parser=parser)
    rows = []
    for e in db.entries:
        rows.append({
            "id":          e.get("ID") or "",
            "type":        e.get("ENTRYTYPE") or "",
            "year":        _year_from_anything(e.get("year")),
            "title":       _clean(e.get("title")),
            "abstract":    _clean(e.get("abstract")),
            "keywords":    _clean(e.get("keywords")),
            "journal":     _clean(e.get("journal") or e.get("booktitle")),
            "author":      _clean(e.get("author")),
            "doi":         _clean(e.get("doi")),
            "subreddit": "", "permalink": "", "timestamp": None,
            "source_file": path.name,
        })
    return rows


def _load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.strip()
            if not line: continue
            try: rows.append(_record_from_dict(json.loads(line), path.name))
            except json.JSONDecodeError: continue
    return rows


def _load_json(path):
    """Stream-parse so multi-GB arrays don't OOM."""
    import ijson
    with open(path, "rb") as fh:
        head = b""
        while True:
            c = fh.read(1)
            if not c: break
            if c not in b" \t\r\n":
                head = c; break
    if head == b"[":
        with open(path, "rb") as fh:
            return [_record_from_dict(d, path.name)
                    for d in ijson.items(fh, "item") if isinstance(d, dict)]
    if head == b"{":
        for prefix in ("data.item", "entries.item", "records.item",
                       "rows.item", "items.item", "posts.item", "comments.item"):
            try:
                with open(path, "rb") as fh:
                    out = [_record_from_dict(d, path.name)
                           for d in ijson.items(fh, prefix) if isinstance(d, dict)]
                if out: return out
            except Exception:
                continue
        with open(path, "r", encoding="utf-8", errors="replace") as fh:
            data = json.load(fh)
        return [_record_from_dict(data, path.name)] if isinstance(data, dict) else []
    return []


def _load_tabular(path, sep):
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as fh:
        for r in csv.DictReader(fh, delimiter=sep):
            rows.append(_record_from_dict(r, path.name))
    return rows


def _load_text(path):
    return [_record_from_text(path.read_text(encoding="utf-8", errors="replace"),
                              path.name)]

def _load_docx(path):
    doc = _Docx(str(path))
    return [_record_from_text("\n".join(p.text for p in doc.paragraphs), path.name)]

def _load_pdf(path):
    reader = _PdfReader(str(path))
    txt = "\n".join((p.extract_text() or "") for p in reader.pages)
    return [_record_from_text(txt, path.name)]


LOADERS = {
    ".bib":    _load_bib, ".json":   _load_json,
    ".jsonl":  _load_jsonl, ".ndjson": _load_jsonl,
    ".csv":    lambda p: _load_tabular(p, ","),
    ".tsv":    lambda p: _load_tabular(p, "\t"),
    ".txt":    _load_text,  ".md":  _load_text,
    ".docx":   _load_docx,  ".pdf": _load_pdf,
}


def discover_files(path, patterns=CORPUS_GLOB_PATTERNS):
    if path.is_file(): return [path]
    if not path.is_dir(): return []
    files = []
    for pat in patterns:
        files.extend(path.rglob(pat))
    seen, uniq = set(), []
    for f in sorted(files):
        if (f in seen or not f.is_file()
            or f.name.startswith(".") or f.name.startswith("~$")
            or "checkpoint" in f.name.lower()):
            continue
        seen.add(f); uniq.append(f)
    return uniq[:CORPUS_MAX_FILES]


def load_corpus(path, files=None, verbose=True):
    target = files if files is not None else discover_files(path)
    rows, skipped = [], []
    for f in target:
        loader = LOADERS.get(f.suffix.lower())
        if not loader:
            skipped.append((f.name, f"unsupported ({f.suffix})")); continue
        try: new = loader(f)
        except Exception as e:
            skipped.append((f.name, f"{type(e).__name__}: {e}")); continue
        rows.extend(new)
        if verbose:
            print(f"   · {f.name:55s} {len(new):>8,} record(s)")
    df = pd.DataFrame(rows, columns=[c for c in CANON_COLS if c != "text_content"])
    if len(df):
        df["text_content"] = (df["title"].fillna("") + " . "
                              + df["abstract"].fillna("") + " . "
                              + df["keywords"].fillna(""))
        df = df[df["text_content"].str.strip().str.len() >= CORPUS_MIN_CHARS] \
               .reset_index(drop=True)
    else:
        df = pd.DataFrame(columns=CANON_COLS)
    return df, skipped

print("✓ Multi-format loader defined")


In [ ]:
# ═══ §2.2 | WRAPPED: corpus load with checkpoint ═══════════════════════
# CHECKPOINT 1 — typically the longest single step (10-15 min for ~30M Reddit records).
# Once cached, subsequent kernel restarts skip this entirely.

if CKPT["raw_corpus"].exists() and not FORCE["raw_corpus"]:
    df_raw = pd.read_parquet(CKPT["raw_corpus"])
    print(f"✓ Resumed: df_raw from {CKPT['raw_corpus'].name}  "
          f"({len(df_raw):,} records, {_fmt_size(CKPT['raw_corpus']).strip()})")
elif CORPUS_PATH.exists():
    print(f"Loading from: {CORPUS_PATH}  (no cache; this will take ~10-15 min)")
    df_raw, skipped = load_corpus(CORPUS_PATH)
    n_files = df_raw["source_file"].nunique() if len(df_raw) else 0
    print(f"\n✓ Loaded {len(df_raw):,} records from {n_files} file(s)")
    if skipped:
        print(f"  Skipped {len(skipped)} file(s); first 5:")
        for n, r in skipped[:5]: print(f"   · {n}: {r}")
    # IMMEDIATE CHECKPOINT — survives any subsequent crash
    df_raw.to_parquet(CKPT["raw_corpus"], index=False)
    print(f"✓ Cached → {CKPT['raw_corpus'].name}  ({_fmt_size(CKPT['raw_corpus']).strip()})")
else:
    print(f"⚠ {CORPUS_PATH} not found — using empty DataFrame.")
    df_raw = pd.DataFrame(columns=CANON_COLS)


In [ ]:
# ═══ §2.3 | SOURCE: NB2 Cell 15 (optional subset selector) ═══
# To scope analysis to one or a few files, edit SUBSET_FILES and re-run.
# Set to None to use the full corpus. Subsetting forces re-load + re-cache.

SUBSET_FILES = None   # e.g. ["r_austin_comments.json", "r_news_comments_TX.json"]

if SUBSET_FILES:
    files = [(CORPUS_PATH / f) if not Path(f).is_absolute() else Path(f)
             for f in SUBSET_FILES]
    missing = [f for f in files if not f.exists()]
    if missing:
        raise FileNotFoundError(f"Subset files not found: {missing}")
    df_raw, _ = load_corpus(CORPUS_PATH, files=files, verbose=True)
    print(f"\n✓ Subset active: {len(df_raw):,} records from {len(files)} file(s)")
    print("  (subsetting bypasses the raw_corpus cache for this session)")
else:
    print("(no subset — full corpus in use)")


In [ ]:
# ═══ §2.4 | NEW: pyarrow string backend (5× faster regex downstream) ═══
# Cheap one-time conversion. Makes §3.2 group scans ~5× faster.
if len(df_raw) and df_raw["text_content"].dtype != "string[pyarrow]":
    t0 = time.time()
    df_raw["text_content"] = df_raw["text_content"].astype("string[pyarrow]")
    print(f"✓ text_content → pyarrow strings  ({time.time()-t0:.1f}s)")
else:
    cur = df_raw["text_content"].dtype if len(df_raw) else "empty"
    print(f"  text_content already {cur}")


## 3. Boolean corpus partitioning

The lead author's keyword groups + boolean searches are the analytical lens. A search's `requirements` is a list where:

- a bare group name (e.g. `"NC"`) means **AND** — the doc must hit at least one term in that group, and
- a bracketed list (e.g. `["Govppl","Agency"]`) means **OR** — at least one of those groups must match.

§3.2 has **two stages**, each checkpointed independently:

- **§3.2a** — one regex scan per keyword group → 16 boolean columns. Slow (~10–15 min with pyarrow). Saves after **every group** so a timeout never loses more than one group's work.
- **§3.2b** — combine groups into boolean-search results, filter to `df`. Seconds.

In [ ]:
# ═══ §3.1 | SOURCE: SB Cell 14 (lead author's _GROUP_REGISTRY + BOOLEAN_SEARCHES) ═══
# Preserved verbatim from Semantic Bridge. Edit groups/searches HERE, not below.

_GROUP_REGISTRY = {
    "NC":      ["NC", "north carolina","western carolina", "WNC", "western appalachia", "appalachia", "buncombe co", "BC", "catawba co",
                "rutherford co", "haywood co", "asheville", "Montreat", "Chimney Rock", "cherokee indian", "cherokee tribe",
                "Sep 26", "september 26", "9/26", "9-26"],
    "TX":      ["TX", "Texas", "CTX", "central texas", "hill country", "kerr co", "kendall co", "travis co", "kerrville", "comfort", "hunt", "ingram",
                "center point", "san saba", "menard", "mason", "bumble bee hills", "july 4", "7/4", "7-4"],
    "Flood":   ["flood", "flash flood", "inundation","wall of water", "tropical storm","inland tsunami", "flooding", "floodplain", "hurricane",
                "guadalupe", "Cypress creek", "mother nature", "God-like",
                "north fork", "south fork", "edmunson creek", "flash flood alley","Swannanoa", "French Broad", "catawba", "winkler creek", "helene"],
    "Radar":   ["Rainmaker", "severe rainfall", "severe thunderstorm", "weather radar system", "weather station", "weather forecast", "category 4",
                "category four", "100-year", "500-year", "alert", "flood warning", "flood watch", "notification", "message", "WARN-CTX", "IPAWS",
                "weather weapon", "cloud seeding", "weather manipulation", "weather machine","climate hoax", "control the weather"],
    "Hoax":    ["conspiracy", "misinformation", "disinformation","fake", "false", "inaccurate", "wrong", "lying", "rumor", "lie", "misleading",
                "propaganda", "rhetoric", "mainstream", "anti-american","geo-engineered weapons", "AI-generated", "AI slop", "scam",
                "migrants", "God's plan", "nothing they could have done",
                "weird cult", "food confiscation", "never happened before"],
    "Infra":   ["I-40", "I-26", "I-39", "impassable", "power outage","lost power", "no power","lack of cell service", "no internet connection",
                "no signal", "no cell service", "no water", "burst pipe", "boil water advisory","wastewater overflow", "SSO", "sewer overflow",
                "sewer runoff", "bridge", "low-water crossing"],
    "Natsys":  ["landslide", "erosion", "debris", "cypress tree", "damming", "stacking up", "water contamination", "water quality",
                "sediment buildup", "vegetation"],
    "Health":  ["PTSD", "chronic stress","heart attack", "suicide", "mold", "mildew", "disease", "illness", "infection", "asthma", "premature death",
                "anxiety", "depression", "hospital",
                "healthcare desert", "IV shortage", "displacement", "therapy"],
    "Agency":  ["FEMA", "NOAA", "NWS", "National Weather Service", "TDEM", "TPWD", "TWDB", "TxDOT", "NCEM", "DEQ", "NCORR",
                "DPS", "UGRA", "GBRA", "FDNY", "DOGE"],
    "Gov":     ["president", "former president","administration", "lieut gov", "lieutenant governor", "governor", "rep",
                "representative", "senator", "chairman", "mayor", "director", "judge", "liberal",
                "leftist", "extremist", "conservative", "republican", "democrat", "moderate", "maga"],
    "Govppl":  ["joe biden","donald trump", "president trump", "pres trump", "biden", "Chuck Edwards", "Edwards", "Mark Robinson", "robinson",
                "josh stein", "roy cooper", "Gov Cooper", " Gov Abbott", "Greg Abbott", "Abbott","ray howard", "stolarczyk", "joe herring", "dub thomas",
                "william thomas", "larry leitha", "tom moser","rob kelly", "kelly", "william rector", "king", "donna campbell", "campbell",
                "flores", "wes virdell", "virdell", "dick eastland"],
    "Response":["evacuation", "evacuate", "cutting off access", "trapped", "emergency response", "first responder", "search and rescue",
                "missing person", "home cleanup", "municipal repair","no helicopters", "no rescue", "firefighter", "fire department",
                "game warden", "volunteer", "state of emergency", "disaster declaration", "help delayed"],
    "GovAid":  ["disaster relief","blockade", "relief aid", "federal aid", "SBA loan","funding", "apply", "application",
                "TSA Program","money", "flood insurance", "LOMR", "LOMA", "national cuts", "budget cuts", "waste"],
    "Media":   ["social media", "podcast", "twitter", "retweet", "X Community", "chatroom", "insta", "instagram", "instagram group",
                "instagram post", "subreddit post", "reddit thread", "subreddit thread", "truth social","YouTube", "fb post", "facebook group",
                "facebook post", "tucker carlson", "fb comment", "facebook comment",
                "alex jones", "joe rogan", "charlie kirk", "shellenberger", "kirk", "rogan", "influencer", "super spreader", "clout chaser"],
    "News":    ["fox", "fox news", "new york times", "NYT", "CNN", "abc", "nbc", "npr", "cbs", "breitbart", "national review",
                "washington times", "turning point", "prageru", "saw on TV", "news article", "saw online"],
    "Tourist": ["la junta", "Mo Ranch", "camp mystic", "Heart O' the Hills", "senior hill", "bubble inn", "eastland",
                "Mystic Cypress Lake", "Heaven's 27", "mystic lawsuit", "tourist", "outsider", "RV parks", "Blue Oak RV", "casa bonita",
                "tourism", "airbnb", "vrbo", "rental"],
}

BOOLEAN_SEARCHES = [
    {"name": "GOWNC",       "event": "NC_Helene", "requirements": ["NC", "Flood", "Radar", "Gov", ["Govppl", "Agency"]]},
    {"name": "GovMisNC",    "event": "NC_Helene", "requirements": ["NC", "Flood", "Radar", "Hoax", ["Gov", "Govppl", "Agency"]]},
    {"name": "MediaMisNC",  "event": "NC_Helene", "requirements": ["NC", "Media", "Hoax", ["Flood", "Radar"]]},
    {"name": "NewsMisNC",   "event": "NC_Helene", "requirements": ["NC", "News",  "Hoax", ["Flood", "Radar"]]},
    {"name": "GovAidNC",    "event": "NC_Helene", "requirements": ["NC", "Flood", ["Infra", "Natsys", "Tourist"], ["GovAid", "Response"]]},
    {"name": "RespMisNC",   "event": "NC_Helene", "requirements": ["NC", "Flood", ["Response", "GovAid"], "Hoax"]},
    {"name": "DisMisNC",    "event": "NC_Helene", "requirements": ["NC", "Flood", ["Infra", "Natsys"], "Hoax"]},
    {"name": "EPHImpactsNC","event": "NC_Helene", "requirements": ["NC", "Flood", "Health", ["Infra", "Natsys"]]},
    {"name": "EPHMisNC",    "event": "NC_Helene", "requirements": ["NC", "Flood", "Health", "Hoax"]},
    # Texas Floods (CTX) — extend in the same shape
]

print(f"✓ {len(_GROUP_REGISTRY)} keyword groups, {len(BOOLEAN_SEARCHES)} boolean searches loaded")


In [ ]:
# ═══ §3.2a | WRAPPED: group scans with per-group checkpointing ═══
# CHECKPOINT 2 — the longest single step on a fresh run. Each completed
# group is saved to disk WITHIN SECONDS. If the session times out
# mid-scan, the next run resumes from the next undone group.

# Determine which groups are already cached
done_groups = set()
if CKPT["group_scans"].exists() and not FORCE["group_scans"]:
    cached = pd.read_parquet(CKPT["group_scans"])
    cached_cols = [c for c in cached.columns if c.startswith("grp_")]
    done_groups = {c[4:] for c in cached_cols}
    # Merge cached columns into df_raw
    if cached_cols:
        df_raw = df_raw.merge(cached[["id","source_file"] + cached_cols],
                              on=["id","source_file"], how="left")
    print(f"✓ Resumed: {len(done_groups)} of {len(_GROUP_REGISTRY)} groups cached")
    if done_groups:
        print(f"  Done: {sorted(done_groups)}")

# Compile patterns only for groups still to scan
_remaining = {name: terms for name, terms in _GROUP_REGISTRY.items()
              if name not in done_groups}

if _remaining:
    _group_patterns = {
        name: re.compile(r"|".join(re.escape(t) for t in terms), re.IGNORECASE)
        for name, terms in _remaining.items()
    }
    est_min = len(_group_patterns) * 1   # rough: ~1 min/group with pyarrow strings
    print(f"\nScanning {len(_group_patterns)} remaining group(s) "
          f"(~{est_min}-{est_min*3} min estimated):")
    t0_total = time.time()
    for i, (name, pat) in enumerate(_group_patterns.items(), 1):
        t0 = time.time()
        df_raw[f"grp_{name}"] = df_raw["text_content"].str.contains(
            pat, regex=True, na=False)
        elapsed = time.time() - t0
        n_hits = int(df_raw[f"grp_{name}"].sum())
        print(f"  [{i:>2}/{len(_group_patterns)}] grp_{name:<10s} "
              f"{elapsed:>6.1f}s   {n_hits:>10,} hits")
        # ── PER-GROUP CHECKPOINT — saves within 1 second ──
        grp_cols = [c for c in df_raw.columns if c.startswith("grp_")]
        df_raw[["id","source_file"] + grp_cols].to_parquet(
            CKPT["group_scans"], index=False)
    print(f"\n✓ All {len(_GROUP_REGISTRY)} groups done in "
          f"{(time.time()-t0_total)/60:.1f} min")
    print(f"✓ Cached → {CKPT['group_scans'].name}  "
          f"({_fmt_size(CKPT['group_scans']).strip()})")
else:
    print(f"✓ All {len(_GROUP_REGISTRY)} groups already cached — nothing to do")


In [ ]:
# ═══ §3.2b | WRAPPED: boolean reduction + matched-df checkpoint ═══
# CHECKPOINT 3 — cheap (seconds) but the result is the analytical input
# for §4-§7, so worth caching.

if CKPT["matched_df"].exists() and not FORCE["matched_df"]:
    df = pd.read_parquet(CKPT["matched_df"])
    match_summary = pd.read_csv(CKPT["match_summary"])
    # Rehydrate list columns from pipe-strings
    for c in ("boolean_matches", "events"):
        if c in df.columns and df[c].dtype == object:
            df[c] = df[c].fillna("").apply(lambda s: s.split("|") if s else [])
    print(f"✓ Resumed: df ({len(df):,} matched docs), match_summary "
          f"({len(match_summary)} searches)")
else:
    def _req_mask(req, frame):
        if isinstance(req, list):
            cols = [f"grp_{g}" for g in req if f"grp_{g}" in frame.columns]
            if not cols: return pd.Series(False, index=frame.index)
            return frame[cols].any(axis=1)
        col = f"grp_{req}"
        if col not in frame.columns: return pd.Series(False, index=frame.index)
        return frame[col]

    for bs in BOOLEAN_SEARCHES:
        mask = pd.Series(True, index=df_raw.index)
        for req in bs["requirements"]:
            mask &= _req_mask(req, df_raw)
        df_raw[f"bool_{bs['name']}"] = mask

    _bool_cols = [f"bool_{bs['name']}" for bs in BOOLEAN_SEARCHES]
    df_raw["boolean_matches"] = df_raw[_bool_cols].apply(
        lambda r: [bs["name"] for bs, hit in zip(BOOLEAN_SEARCHES, r) if hit], axis=1)
    df_raw["events"] = df_raw["boolean_matches"].apply(
        lambda names: sorted({bs["event"] for bs in BOOLEAN_SEARCHES
                              if bs["name"] in names and bs["event"] != "Multi"}))
    df_raw["any_match"] = df_raw["boolean_matches"].str.len() > 0

    df = df_raw[df_raw["any_match"]].copy().reset_index(drop=True)

    match_summary = pd.DataFrame({
        "boolean_search": [bs["name"] for bs in BOOLEAN_SEARCHES],
        "event":          [bs["event"] for bs in BOOLEAN_SEARCHES],
        "matches":        [int(df_raw[f"bool_{bs['name']}"].sum()) for bs in BOOLEAN_SEARCHES],
    })

    # Save: flatten list columns to pipe-strings for parquet round-trip
    save_df = df.copy()
    save_df["boolean_matches"] = save_df["boolean_matches"].apply(lambda lst: "|".join(lst))
    save_df["events"] = save_df["events"].apply(lambda lst: "|".join(lst))
    save_df.to_parquet(CKPT["matched_df"], index=False)
    match_summary.to_csv(CKPT["match_summary"], index=False)

    print(f"✓ {len(df):,} of {len(df_raw):,} docs matched "
          f"({len(df)/max(len(df_raw),1)*100:.1f}%)")
    print(f"✓ Cached → {CKPT['matched_df'].name}, {CKPT['match_summary'].name}")

match_summary


## 4. Sub-topic discovery within boolean partitions

LDA per boolean partition (with TF-IDF fallback for small partitions). Stopwords augmented with subreddit/author names to suppress metadata leakage. **Checkpointed** — the result `subtopics_df` is saved to disk and reloaded on re-run.

In [ ]:
# ═══ §4.1 | SOURCE: SB Cell 20 (metadata-aware stopwords) ═══
STOPWORDS_BASE = {
    "a","about","above","after","again","all","am","an","and","any","are","as","at",
    "be","because","been","before","being","below","between","both","but","by","can",
    "did","do","does","doing","don","down","during","each","few","for","from","further",
    "had","has","have","having","he","her","here","hers","herself","him","himself","his",
    "how","i","if","in","into","is","it","its","itself","just","let","me","more","most",
    "my","myself","no","nor","not","now","of","off","on","once","only","or","other","our",
    "ours","ourselves","out","over","own","s","same","she","should","so","some","such","t",
    "than","that","the","their","theirs","them","themselves","then","there","these","they",
    "this","those","through","to","too","under","until","up","very","was","we","were",
    "what","when","where","which","while","who","whom","why","will","with","you","your",
    "yours","yourself","yourselves",
    "http","https","www","com","reddit","imgur","edit","deleted","removed","nbsp","amp",
    "like","know","think","going","really","would","could","got","get","just",
    "yeah","okay","um","uh","oh","actually","basically","literally","lol",
    "war","archive","trump","russian","sumerian","epstein",
}

def build_stopwords(frame):
    sw = set(STOPWORDS_BASE)
    if "subreddit" in frame.columns:
        sw.update(str(s).lower() for s in frame["subreddit"].dropna().unique() if s)
    if "author" in frame.columns:
        sw.update(str(a).lower().lstrip("u_") for a in frame["author"].dropna().unique() if a)
    sw.discard("")
    return sw

CONTENT_STOPWORDS = build_stopwords(df)
print(f"✓ {len(CONTENT_STOPWORDS):,} stopwords")


In [ ]:
# ═══ §4.2 | WRAPPED: LDA per partition with checkpoint ═══
# CHECKPOINT 4 — for the multi-event Reddit corpus, LDA across ~10
# partitions takes a few minutes. Cached to skip on re-run.

SUBTOPICS_PER_CORPUS = 3
MIN_DOCS_FOR_LDA     = 25

def top_terms_tfidf(texts, stopwords, n=10):
    vec = TfidfVectorizer(max_features=2000, stop_words=list(stopwords),
                          ngram_range=(1, 2), min_df=2)
    try: X = vec.fit_transform(texts)
    except ValueError: return []
    means = np.asarray(X.mean(axis=0)).ravel()
    vocab = np.array(vec.get_feature_names_out())
    return list(vocab[means.argsort()[::-1][:n]])

def lda_subtopics(texts, stopwords, k, n_terms=10):
    vec = CountVectorizer(max_features=2000, stop_words=list(stopwords),
                          ngram_range=(1, 2), min_df=2)
    try: X = vec.fit_transform(texts)
    except ValueError: return []
    lda = LatentDirichletAllocation(n_components=k, random_state=42,
                                    learning_method="batch", max_iter=20)
    lda.fit(X)
    vocab = np.array(vec.get_feature_names_out())
    return [list(vocab[comp.argsort()[::-1][:n_terms]]) for comp in lda.components_]

if CKPT["subtopics"].exists() and not FORCE["subtopics"]:
    subtopics_df = pd.read_parquet(CKPT["subtopics"])
    # Rehydrate top_terms list column from pipe-string
    if len(subtopics_df) and isinstance(subtopics_df["top_terms"].iloc[0], str):
        subtopics_df["top_terms"] = subtopics_df["top_terms"].apply(
            lambda s: s.split("|") if s else [])
    subtopics_df["top_terms_str"] = subtopics_df["top_terms"].str.join(", ")
    print(f"✓ Resumed: {len(subtopics_df)} sub-topics from {CKPT['subtopics'].name}")
else:
    subtopic_rows = []
    print(f"Computing LDA sub-topics across {len(BOOLEAN_SEARCHES)} boolean partitions:")
    for i, bs in enumerate(BOOLEAN_SEARCHES, 1):
        sub = df[df[f"bool_{bs['name']}"]]
        if len(sub) == 0:
            print(f"  [{i:>2}/{len(BOOLEAN_SEARCHES)}] {bs['name']:<14s} 0 docs — skipped")
            continue
        if len(sub) >= MIN_DOCS_FOR_LDA:
            terms_list = lda_subtopics(sub["text_content"].tolist(),
                                       CONTENT_STOPWORDS, SUBTOPICS_PER_CORPUS)
            for j, terms in enumerate(terms_list):
                subtopic_rows.append({"boolean_search": bs["name"], "event": bs["event"],
                                      "subtopic_id": j, "n_docs": len(sub),
                                      "method": "LDA", "top_terms": terms})
            print(f"  [{i:>2}/{len(BOOLEAN_SEARCHES)}] {bs['name']:<14s} "
                  f"{len(sub):>6,} docs → {len(terms_list)} LDA sub-topics")
        else:
            subtopic_rows.append({"boolean_search": bs["name"], "event": bs["event"],
                                  "subtopic_id": 0, "n_docs": len(sub), "method": "TF-IDF",
                                  "top_terms": top_terms_tfidf(sub["text_content"].tolist(),
                                                               CONTENT_STOPWORDS)})
            print(f"  [{i:>2}/{len(BOOLEAN_SEARCHES)}] {bs['name']:<14s} "
                  f"{len(sub):>6,} docs → 1 TF-IDF set (below LDA threshold)")

    subtopics_df = pd.DataFrame(subtopic_rows)
    if len(subtopics_df):
        subtopics_df["top_terms_str"] = subtopics_df["top_terms"].str.join(", ")
        # Save: flatten top_terms to pipe-string for parquet
        save_df = subtopics_df.copy()
        save_df["top_terms"] = save_df["top_terms"].apply(lambda lst: "|".join(lst))
        save_df.to_parquet(CKPT["subtopics"], index=False)
        print(f"\n✓ {len(subtopics_df)} sub-topics → cached to {CKPT['subtopics'].name}")

subtopics_df[["boolean_search","event","subtopic_id","n_docs","method","top_terms_str"]] \
    if len(subtopics_df) else subtopics_df


In [ ]:
# ═══ §4.3 | WRAPPED: optional LLM topic labeling with checkpoint ═══
# Disabled by default. To enable: export ANTHROPIC_API_KEY, then set
# ENABLE_LLM_LABELS = True. Labels are checkpointed because LLM calls
# cost real money and shouldn't be re-run on session restart.

ENABLE_LLM_LABELS = False
LLM_MODEL         = "claude-opus-4-7"

def llm_label_topic(top_terms, examples):
    import os
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("ANTHROPIC_API_KEY is not set.")
    try: from anthropic import Anthropic
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anthropic"])
        from anthropic import Anthropic
    client = Anthropic()
    prompt = (
        "You are labeling a sub-topic extracted from a Reddit corpus about natural disaster events.\n"
        f"Top terms: {', '.join(top_terms)}\n\n"
        "Three example snippets:\n" + "\n---\n".join(examples[:3]) +
        "\n\nReturn ONLY a 3–7 word topic label. No preamble, no quotation marks."
    )
    resp = client.messages.create(model=LLM_MODEL, max_tokens=40,
                                  messages=[{"role": "user", "content": prompt}])
    return resp.content[0].text.strip()

if len(subtopics_df):
    subtopics_df["expert_triaged"] = False
    if CKPT["llm_labels"].exists() and not FORCE["llm_labels"]:
        cached = pd.read_csv(CKPT["llm_labels"])
        subtopics_df["llm_label_provisional"] = cached["llm_label_provisional"]
        print(f"✓ Resumed: {len(cached)} LLM labels from {CKPT['llm_labels'].name}")
    elif ENABLE_LLM_LABELS:
        labels = []
        for _, row in subtopics_df.iterrows():
            sub = df[df[f"bool_{row['boolean_search']}"]]
            examples = sub["text_content"].head(3).tolist()
            try: labels.append(llm_label_topic(row["top_terms"], examples))
            except Exception as e: labels.append(f"[label-failed: {type(e).__name__}]")
        subtopics_df["llm_label_provisional"] = labels
        pd.DataFrame({"boolean_search": subtopics_df["boolean_search"],
                      "subtopic_id":    subtopics_df["subtopic_id"],
                      "llm_label_provisional": labels}) \
          .to_csv(CKPT["llm_labels"], index=False)
        print(f"✓ Generated {sum(1 for l in labels if not l.startswith('['))} labels "
              f"→ cached to {CKPT['llm_labels'].name}")
    else:
        subtopics_df["llm_label_provisional"] = ""
        print("  LLM labeling disabled — top_terms serve as sub-topic identifiers.")


## 5. Science backbone & network

The backbone projects each sub-topic onto scientific domains and subdisciplines. **Two simple choices** — pick one in §5.1:

| Choice | What it does | When to use |
|---|---|---|
| **ETO Map of Science CSV** | Build the backbone from your ETO cluster export. ~10 domains, up to 200 subdisciplines drawn from your query. | Production analysis. Reproducible — anyone with the CSV gets the same backbone. |
| **Inline scaffold** | Use the built-in 7-domain misinformation backbone in `INLINE_BACKBONE`. | Quick prototyping, demos, or when you don't have an ETO export. |

The notebook auto-selects: if `ETO_EXPORT_PATH` is set *and* the file exists, the ETO loader runs; otherwise the inline scaffold takes over. To swap in an updated ETO export, change the path and re-run §5.1 — nothing else needs to change.

No `semantic_bridge_pipeline` install needed; the ETO loader is self-contained.

In [ ]:
# ═══ §5.1 | Science backbone — ETO export OR inline scaffold ═══════════
#
# The backbone projects sub-topics onto scientific domains and subdisciplines.
# Two simple choices:
#
#   • Provide an ETO Map-of-Science CSV path below     → richer, current
#   • Set ETO_EXPORT_PATH = None                       → inline 7-domain scaffold
#
# That's it. The notebook handles the rest. You can swap in an updated
# ETO export at any time by changing the path and re-running this cell.

# ── Backbone source (edit this one line) ──────────────────────────────
ETO_EXPORT_PATH = Path("/work/01813/sawp33/ScienceBackbone/eto-mediamisinfo-map-of-science.csv")

# ── Optional tuning (only used when ETO_EXPORT_PATH is set) ───────────
ETO_MAX_CLUSTERS = 200    # cap on clusters loaded (keeps network viz readable)
ETO_MIN_GROWTH   = 0      # only load clusters with growth_rating ≥ this (0-100)
ETO_REQUIRED     = False  # if True and ETO can't be loaded, raise instead of falling back


# ── ETO loader (self-contained — no external dependencies) ────────────
def build_backbone_from_eto_csv(path, max_clusters=200, min_growth=0):
    """Build a backbone dict from an ETO Map of Science cluster CSV.

    Expected columns: 'Cluster ID', 'Cluster title', 'Cluster summary',
    'Most common research field', 'Cluster size', 'Growth rating'.

    Returns: {domain: {subdisciplines: [...], terms: [...]}, ...}
    """
    _STOP = {"the","a","an","and","or","of","in","on","for","with","to","from","by",
             "as","at","is","are","was","were","be","been","being","this","that",
             "between","through","using","based","via","into","over","other","new"}
    eto = pd.read_csv(path, encoding="utf-8-sig")
    eto.columns = [c.strip().lower().replace(" ", "_") for c in eto.columns]
    required = {"cluster_title", "most_common_research_field", "cluster_size", "growth_rating"}
    missing = required - set(eto.columns)
    if missing:
        raise ValueError(f"ETO CSV missing columns: {missing}. "
                         f"Got: {list(eto.columns)}")
    eto["growth_rating"] = pd.to_numeric(eto["growth_rating"], errors="coerce").fillna(0)
    eto["cluster_size"]  = pd.to_numeric(eto["cluster_size"],  errors="coerce").fillna(0)
    eto = eto[eto["growth_rating"] >= min_growth].nlargest(max_clusters, "cluster_size")
    backbone = {}
    for field, group in eto.groupby("most_common_research_field"):
        if not field or pd.isna(field): continue
        domain = str(field).title()
        subs = group["cluster_title"].astype(str).tolist()
        terms = set()
        for _, row in group.iterrows():
            title = str(row["cluster_title"]).lower()
            terms.add(title)
            for w in re.findall(r"[a-z][a-z\-]{3,}", title):
                if w not in _STOP: terms.add(w)
        backbone[domain] = {"subdisciplines": subs, "terms": sorted(terms)[:150]}
    return backbone


# ── Inline fallback backbone (used if ETO_EXPORT_PATH is None or missing) ──
INLINE_BACKBONE = {
    "Atmospheric Science":       {"subdisciplines": ["Meteorology", "Climate Science", "Severe Weather"],
                                  "terms": ["wind","storm","precipitation","forecast","climate","atmosphere","warming","NOAA"]},
    "Riverine Flash Flooding":   {"subdisciplines": ["Flood Hydrology", "Water Resources", "Debris Damage"],
                                  "terms": ["flood","river","water","rainfall","runoff","basin","watershed","dam","levee",
                                            "landslide","mudslide","drought","erosion","debris","guadalupe","fork","french broad","swannanoa"]},
    "Emergency Management":      {"subdisciplines": ["Response", "Evacuation", "Recovery"],
                                  "terms": ["evacuation","rescue","relief","emergency","response","shelter","FEMA","aid"]},
    "Public Health":             {"subdisciplines": ["Environmental Health", "Disaster Mental Health"],
                                  "terms": ["health","injury","mold","water quality","mental","trauma","grief","anxiety","contamination"]},
    "Infrastructure Engineering":{"subdisciplines": ["Power", "Transportation", "Communications"],
                                  "terms": ["power","bridge","road","low-water crossing","network","outage","cell",
                                            "internet","infrastructure","repair","access"]},
    "Political Communication":   {"subdisciplines": ["Government Messaging", "Social Media Discourse"],
                                  "terms": ["president","governor","administration","biden","trump","FEMA","abbott","NWS","liberal",
                                            "maga","republican","UGRA"]},
    "Information Integrity":     {"subdisciplines": ["Misinformation", "Disinformation"],
                                  "terms": ["conspiracy","hoax","fake","misinformation","disinformation","weather weapon","cloud seeding"]},
}


# ── Resolve which backbone to use ─────────────────────────────────────
SCIENCE_BACKBONE = None
BACKBONE_SOURCE  = None

if ETO_EXPORT_PATH and Path(ETO_EXPORT_PATH).exists():
    try:
        SCIENCE_BACKBONE = build_backbone_from_eto_csv(
            ETO_EXPORT_PATH,
            max_clusters=ETO_MAX_CLUSTERS,
            min_growth=ETO_MIN_GROWTH,
        )
        BACKBONE_SOURCE = (f"ETO export: {Path(ETO_EXPORT_PATH).name} "
                           f"({len(SCIENCE_BACKBONE)} domains, top {ETO_MAX_CLUSTERS} clusters)")
    except Exception as e:
        print(f"⚠ Could not load ETO export: {type(e).__name__}: {e}")
        if ETO_REQUIRED:
            raise
        print(f"  Falling back to inline backbone.")
elif ETO_EXPORT_PATH:
    print(f"⚠ ETO file not found at {ETO_EXPORT_PATH}")
    if ETO_REQUIRED:
        raise FileNotFoundError(f"ETO_REQUIRED is True but file not found: {ETO_EXPORT_PATH}")
    print(f"  Falling back to inline backbone.")

if SCIENCE_BACKBONE is None:
    SCIENCE_BACKBONE = INLINE_BACKBONE
    BACKBONE_SOURCE  = f"Inline misinformation-focused scaffold ({len(INLINE_BACKBONE)} domains)"


# ── Backbone mapper (unchanged) ───────────────────────────────────────
def map_terms_to_backbone(terms, backbone):
    terms_l = [t.lower() for t in terms]
    hits = []
    for domain, node in backbone.items():
        if not isinstance(node, dict): continue
        subs = node.get("subdisciplines") or node.get("subs") or ["General"]
        node_terms = node.get("terms") or node.get("keywords") or [domain.lower()]
        for sub in subs:
            score = sum(1 for t in terms_l if any(str(kw).lower() in t for kw in node_terms))
            if score > 0:
                hits.append((domain, sub, score))
    if not hits:
        return [("Uncategorized", "General", 0)]
    return sorted(hits, key=lambda x: -x[2])[:3]

if len(subtopics_df):
    subtopics_df["backbone_mapping"] = subtopics_df["top_terms"].apply(
        lambda ts: map_terms_to_backbone(ts, SCIENCE_BACKBONE))
    subtopics_df["primary_domain"] = subtopics_df["backbone_mapping"].apply(lambda h: h[0][0])

print(f"✓ Backbone: {BACKBONE_SOURCE}")
n_subs = sum(len(n.get("subdisciplines", [])) for n in SCIENCE_BACKBONE.values())
print(f"  Domains: {list(SCIENCE_BACKBONE.keys())}")
print(f"  Total subdisciplines: {n_subs}")
if len(subtopics_df):
    print("\nSub-topics by primary domain:")
    print(subtopics_df.groupby("primary_domain").size().sort_values(ascending=False).to_string())


In [ ]:
# ═══ §5.2 | SOURCE: SB Cell 29 (boolean → backbone network) ═══
def _node_subs(node):
    if not isinstance(node, dict): return ["General"]
    return node.get("subdisciplines") or node.get("subs") or ["General"]

def build_network(subtopics_df, backbone):
    G = nx.Graph()
    for domain, node in backbone.items():
        G.add_node(domain, kind="domain")
        for sub in _node_subs(node):
            G.add_node(sub, kind="subdiscipline")
            G.add_edge(domain, sub, weight=1)
    for bs_name, grp in subtopics_df.groupby("boolean_search"):
        event = grp["event"].iloc[0]
        G.add_node(bs_name, kind="boolean", event=event)
        for _, row in grp.iterrows():
            for domain, sub, score in row["backbone_mapping"]:
                if sub in G:
                    w = G[bs_name].get(sub, {}).get("weight", 0) + score
                    G.add_edge(bs_name, sub, weight=w)
    return G

EVENT_COLORS = {"TX_Flood": "#BF5700", "NC_Helene": "#005f73",
                "LA_Fires": "#c1121f", "Multi": "#6a4c93"}

if len(subtopics_df):
    G = build_network(subtopics_df, SCIENCE_BACKBONE)
    pos = nx.spring_layout(G, seed=42, k=0.8, iterations=100)
    KIND_COLORS = {"domain": "#1f77b4", "subdiscipline": "#8c8c8c"}

    edge_x, edge_y = [], []
    for u, v in G.edges():
        edge_x += [pos[u][0], pos[v][0], None]
        edge_y += [pos[u][1], pos[v][1], None]

    traces = [go.Scatter(x=edge_x, y=edge_y, mode="lines",
                         line=dict(color="#d0d0d0", width=0.8),
                         hoverinfo="none", showlegend=False)]
    for kind in ["domain", "subdiscipline"]:
        nodes = [n for n, d in G.nodes(data=True) if d.get("kind") == kind]
        traces.append(go.Scatter(
            x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes],
            mode="markers+text",
            marker=dict(size=22 if kind=="domain" else 14,
                        color=KIND_COLORS[kind], line=dict(color="white", width=1)),
            text=nodes, textposition="top center", textfont=dict(size=9),
            hoverinfo="text", name=kind.capitalize()))
    for event, color in EVENT_COLORS.items():
        nodes = [n for n, d in G.nodes(data=True)
                 if d.get("kind") == "boolean" and d.get("event") == event]
        if not nodes: continue
        traces.append(go.Scatter(
            x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes],
            mode="markers+text",
            marker=dict(size=18, color=color, symbol="diamond",
                        line=dict(color="white", width=1)),
            text=nodes, textposition="bottom center",
            textfont=dict(size=9, color=color),
            hoverinfo="text", name=f"Boolean: {event}"))

    fig_network = go.Figure(traces, layout=go.Layout(
        title="Boolean Searches → Science Backbone",
        showlegend=True, hovermode="closest", height=700, plot_bgcolor="white",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig_network.write_html(str(OUTPUT_DIR / "network_boolean_backbone.html"),
                           include_plotlyjs="cdn")
    fig_network.show()


## 6. Burst analysis

Per-event Kleinberg burst detection, anchored at each event's day-zero. **Per-event checkpointing** — if a session times out mid-event, finished events are kept and the next run picks up the unfinished ones.

In [ ]:
# ═══ §6.1 | SOURCE: SB Cell 32 (Kleinberg 2-state burst detector) ═══
def kleinberg_bursts(timestamps, s=2.0, gamma=1.0):
    if len(timestamps) < 4: return []
    times = sorted(timestamps)
    gaps = [(times[i+1] - times[i]).total_seconds() / 86400.0
            for i in range(len(times) - 1)]
    gaps = [max(g, 1e-6) for g in gaps]
    mean_gap = np.mean(gaps)
    if mean_gap <= 0: return []
    rates = [1.0 / mean_gap, s / mean_gap]
    n = len(gaps)
    cost = [[0.0, 0.0] for _ in range(n)]
    back = [[0, 0] for _ in range(n)]
    for k in (0, 1):
        cost[0][k] = -np.log(rates[k] * np.exp(-rates[k] * gaps[0]))
    for i in range(1, n):
        for k in (0, 1):
            best, best_prev = None, 0
            for j in (0, 1):
                trans = (k - j) * gamma * np.log(n) if k > j else 0.0
                c = cost[i-1][j] + trans - np.log(rates[k] * np.exp(-rates[k] * gaps[i]))
                if best is None or c < best:
                    best, best_prev = c, j
            cost[i][k] = best
            back[i][k] = best_prev
    path = [0] * n
    path[-1] = 0 if cost[-1][0] <= cost[-1][1] else 1
    for i in range(n-2, -1, -1):
        path[i] = back[i+1][path[i+1]]
    bursts, in_burst, start_idx = [], False, 0
    for i, st in enumerate(path):
        if st == 1 and not in_burst:
            in_burst, start_idx = True, i
        elif st == 0 and in_burst:
            intensity = sum(1/gaps[j] for j in range(start_idx, i)) / max(i - start_idx, 1)
            bursts.append((times[start_idx], times[i], intensity))
            in_burst = False
    if in_burst:
        intensity = sum(1/gaps[j] for j in range(start_idx, n)) / max(n - start_idx, 1)
        bursts.append((times[start_idx], times[-1], intensity))
    return bursts

print("✓ Kleinberg burst detector defined")


In [ ]:
# ═══ §6.2 | WRAPPED: per-event burst heatmaps with per-event checkpoint ═══
# CHECKPOINT 5 — each event saves its matrix + HTML the moment it finishes.
# A timeout in event 2/3 leaves event 1 intact; next run resumes with event 2.

BURST_TERMS_PER_EVENT = 12

def term_timestamps(docs, term):
    mask = docs["text_content"].str.contains(rf"\b{re.escape(term)}\b",
                                              case=False, regex=True, na=False)
    return docs.loc[mask, "timestamp"].dropna().tolist()

def distinctive_terms(sub_df, all_df, n=BURST_TERMS_PER_EVENT):
    in_corpus  = " ".join(sub_df["text_content"].tolist())
    out_corpus = " ".join(all_df.loc[~all_df.index.isin(sub_df.index),
                                     "text_content"].tolist())
    vec = TfidfVectorizer(max_features=3000, stop_words=list(CONTENT_STOPWORDS), min_df=2)
    try: X = vec.fit_transform([in_corpus, out_corpus])
    except ValueError: return []
    diff = (X.toarray()[0] - X.toarray()[1])
    vocab = np.array(vec.get_feature_names_out())
    return list(vocab[diff.argsort()[::-1][:n]])

event_burst_figs = {}
for event, day0_str in EVENT_DAY_ZERO.items():
    event_html  = OUTPUT_DIR / f"burst_{event}.html"
    event_cache = CKPT["burst_data"] / f"{event}.parquet"

    if event_html.exists() and event_cache.exists() and not FORCE["bursts"]:
        print(f"✓ Resumed: {event} (cache + HTML present)")
        continue

    day0 = datetime.fromisoformat(day0_str).replace(tzinfo=timezone.utc)
    win_start = day0 - timedelta(days=WINDOW_DAYS)
    win_end   = day0 + timedelta(days=WINDOW_DAYS)
    sub = df[df["events"].apply(lambda es: event in es) &
             df["timestamp"].between(win_start, win_end)].copy()
    if len(sub) < 10:
        print(f"  ⚠ {event}: only {len(sub)} docs in window — skipping")
        continue
    terms = distinctive_terms(sub, df)
    if not terms: continue

    days = list(range(-WINDOW_DAYS, WINDOW_DAYS + 1))
    matrix = np.zeros((len(terms), len(days)))
    for ti, term in enumerate(terms):
        ts = term_timestamps(sub, term)
        bursts = kleinberg_bursts(ts)
        for b_start, b_end, intensity in bursts:
            d0, d1 = (b_start - day0).days, (b_end - day0).days
            for d in range(max(d0, -WINDOW_DAYS), min(d1, WINDOW_DAYS) + 1):
                matrix[ti, d - days[0]] = max(matrix[ti, d - days[0]], intensity)

    # ── Per-event checkpoint: save BOTH matrix and HTML immediately ──
    pd.DataFrame(matrix, index=terms,
                 columns=[f"day_{d}" for d in days]).to_parquet(event_cache)

    fig = go.Figure(go.Heatmap(z=matrix, x=days, y=terms, colorscale="Oranges",
                               colorbar=dict(title="burst<br>intensity")))
    fig.add_vline(x=0, line=dict(color="red", width=2, dash="dash"),
                  annotation_text=f"Day 0 ({day0_str})", annotation_position="top")
    fig.update_layout(title=f"{event} — term bursts around day-zero (±{WINDOW_DAYS} days)",
                      xaxis_title="Days since day-zero", yaxis_title="Term",
                      height=450, plot_bgcolor="white")
    fig.write_html(str(event_html), include_plotlyjs="cdn")
    event_burst_figs[event] = fig
    fig.show()
    print(f"✓ {event}: {len(sub):,} docs, {len(terms)} terms → cached")


## 7. Cross-filter comparisons

Match counts, aligned-timeline streamgraph, and Jaccard term-overlap. All fast (seconds), so not checkpointed — HTML is overwritten on every run.

In [ ]:
# ═══ §7.1 | SOURCE: SB Cell 37 (match counts bar) ═══
counts_df = match_summary.copy().sort_values("matches", ascending=True)
fig_counts = px.bar(counts_df, x="matches", y="boolean_search", color="event",
                    orientation="h", color_discrete_map=EVENT_COLORS,
                    title="Documents matched per boolean search")
fig_counts.update_layout(height=400, plot_bgcolor="white",
                         yaxis_title="", xaxis_title="Matched documents")
fig_counts.write_html(str(OUTPUT_DIR / "match_counts.html"), include_plotlyjs="cdn")
fig_counts.show()


In [ ]:
# ═══ §7.2 | SOURCE: SB Cell 39 (aligned-timeline streamgraph) ═══
rows = []
for event, day0_str in EVENT_DAY_ZERO.items():
    day0 = datetime.fromisoformat(day0_str).replace(tzinfo=timezone.utc)
    for bs in BOOLEAN_SEARCHES:
        if bs["event"] != event: continue
        sub = df[df[f"bool_{bs['name']}"] & df["timestamp"].between(
            day0 - timedelta(days=WINDOW_DAYS), day0 + timedelta(days=WINDOW_DAYS))]
        if sub.empty: continue
        daily = sub.groupby(sub["timestamp"].dt.date).size().reset_index(name="count")
        daily["days_since_day0"] = daily["timestamp"].apply(
            lambda d: (datetime.combine(d, datetime.min.time(), tzinfo=timezone.utc) - day0).days)
        daily["event"] = event
        daily["boolean_search"] = bs["name"]
        rows.append(daily[["days_since_day0", "count", "event", "boolean_search"]])

if rows:
    aligned = pd.concat(rows, ignore_index=True)
    fig_aligned = px.area(aligned, x="days_since_day0", y="count", color="boolean_search",
                          facet_row="event",
                          title="Aligned timelines: daily volume vs. days-since-day-zero",
                          height=650)
    fig_aligned.add_vline(x=0, line=dict(color="red", width=1, dash="dash"))
    fig_aligned.update_layout(plot_bgcolor="white")
    fig_aligned.write_html(str(OUTPUT_DIR / "aligned_timelines.html"), include_plotlyjs="cdn")
    fig_aligned.show()
else:
    print("  No data in event windows.")


In [ ]:
# ═══ §7.3 | SOURCE: SB Cell 41 (Jaccard term-overlap heatmap) ═══
def top_terms_set(boolean_name, n=30):
    sub = df[df[f"bool_{boolean_name}"]]
    if len(sub) < 3: return set()
    return set(top_terms_tfidf(sub["text_content"].tolist(), CONTENT_STOPWORDS, n=n))

names = [bs["name"] for bs in BOOLEAN_SEARCHES]
term_sets = {n: top_terms_set(n) for n in names}
J = np.zeros((len(names), len(names)))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        u = term_sets[a] | term_sets[b]
        J[i, j] = len(term_sets[a] & term_sets[b]) / len(u) if u else 0.0

fig_jac = go.Figure(go.Heatmap(z=J, x=names, y=names, colorscale="Blues",
                               zmin=0, zmax=1, colorbar=dict(title="Jaccard")))
fig_jac.update_layout(title="Boolean-search vocabulary overlap (top-30 TF-IDF terms)",
                      height=550, xaxis_tickangle=-30, plot_bgcolor="white")
fig_jac.write_html(str(OUTPUT_DIR / "jaccard_similarity.html"), include_plotlyjs="cdn")
fig_jac.show()


## 8. Export

In [ ]:
# ═══ §8.1 | NEW: Export run-stamped final artifacts ═══
# Final exports get a timestamp suffix (so multiple runs don't overwrite).
# The checkpoint files in §1.3 are SEPARATE from these — they're the
# resume-from-disk caches and stay put.

from datetime import datetime as _dt
stamp = _dt.now().strftime("%Y%m%d_%H%M%S")

export_df = df.copy()
if isinstance(export_df["boolean_matches"].iloc[0] if len(export_df) else "", list):
    export_df["boolean_matches"] = export_df["boolean_matches"].apply(lambda lst: "|".join(lst))
    export_df["events"]          = export_df["events"].apply(lambda lst: "|".join(lst))
matched_path = OUTPUT_DIR / f"matched_documents_{stamp}.parquet"
export_df.to_parquet(matched_path, index=False)

subtopics_path = OUTPUT_DIR / f"subtopics_{stamp}.csv"
if len(subtopics_df):
    st_export = subtopics_df.copy()
    st_export["top_terms"] = st_export["top_terms"].apply(lambda lst: "|".join(lst))
    if "backbone_mapping" in st_export.columns:
        st_export["backbone_mapping"] = st_export["backbone_mapping"].apply(
            lambda hits: "|".join(f"{d}>{s}({sc})" for d, s, sc in hits))
    st_export.to_csv(subtopics_path, index=False, encoding="utf-8-sig")

summary_path = OUTPUT_DIR / f"match_summary_{stamp}.csv"
match_summary.to_csv(summary_path, index=False)

report = [
    f"# Media Misinformation — Run {stamp}",
    "",
    f"- **Corpus source**:  `{CORPUS_PATH}`",
    f"- **Total docs ingested**:  {len(df_raw):,}",
    f"- **Docs matching ≥1 boolean search**:  {len(df):,}  ({len(df)/max(len(df_raw),1)*100:.1f}%)",
    f"- **Keyword groups**:  {len(_GROUP_REGISTRY)}",
    f"- **Boolean searches**:  {len(BOOLEAN_SEARCHES)}",
    f"- **Sub-topics discovered**:  {len(subtopics_df)}",
    f"- **Backbone source**:  {BACKBONE_SOURCE}",
    f"- **Events**:  {', '.join(EVENT_DAY_ZERO.keys())}",
    "",
    "## Match counts",
    match_summary.to_markdown(index=False),
    "",
    "## Outputs",
    f"- `{matched_path.name}`  (parquet)",
    f"- `{subtopics_path.name}`",
    f"- `{summary_path.name}`",
    "- `network_boolean_backbone.html`",
    "- `match_counts.html`",
    "- `aligned_timelines.html`",
    "- `jaccard_similarity.html`",
]
for evt in EVENT_DAY_ZERO:
    fp = OUTPUT_DIR / f"burst_{evt}.html"
    if fp.exists(): report.append(f"- `{fp.name}`")

report_path = OUTPUT_DIR / f"report_{stamp}.md"
report_path.write_text("\n".join(report))

print(f"✓ Run complete — final exports in {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob(f"*{stamp}*")):
    print(f"  · {f.name}")


## 9. Recovery & cache management

Two utilities for managing the resumable workflow:

- **§9.1** — inventory of every cached artifact (compute caches *and* final exports)
- **§9.2** — selective reset (clear caches to force recompute of specific stages)

These are *utilities* — they don't need to run for the pipeline to work. Visit them when something looks off, when you want to start a section fresh, or when you want to verify what's on disk before sharing the results.

In [ ]:
# ═══ §9.1 | NEW: Full artifact inventory ═══
# Lists every checkpoint AND every final output, with sizes and dates.

print("="*72)
print("COMPUTE CHECKPOINTS  (skip-if-present logic in §2-§6)")
print("="*72)
print(f"{'Stage':<25} {'Status':<14} {'Size':<10} {'Last modified':<19}")
print("-" * 70)
for label, key in _STAGES:
    p = CKPT[key]
    if p.exists():
        forced = FORCE.get(key, False)
        status = "FORCE-RERUN" if forced else "✓ cached"
        mt_src = p.stat().st_mtime if p.is_file() else max(
            (f.stat().st_mtime for f in p.rglob("*") if f.is_file()), default=0)
        mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(mt_src)) if mt_src else ""
    else:
        status, mtime = "  pending", ""
    print(f"{label:<25} {status:<14} {_fmt_size(p):<10} {mtime:<19}")

print()
print("="*72)
print("FINAL OUTPUTS  (visualizations + run-stamped exports)")
print("="*72)
final_outputs = sorted(p for p in OUTPUT_DIR.iterdir()
                       if p.is_file() and p.suffix in (".html", ".parquet", ".csv", ".md")
                       and "checkpoint" not in str(p.parent))
if final_outputs:
    print(f"{'File':<50} {'Size':<10} {'Modified':<19}")
    print("-" * 80)
    for p in final_outputs:
        mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(p.stat().st_mtime))
        print(f"{p.name:<50} {_fmt_size(p):<10} {mtime:<19}")
else:
    print("  (no final outputs yet — run §8.1)")


In [ ]:
# ═══ §9.2 | NEW: Selective cache reset ═══
# Clears specific compute caches so the corresponding stage(s) re-run
# on the next pass. Safe — does NOT touch your final exports in OUTPUT_DIR.
#
# To use:
#   1. Edit RESET_STAGES to the list of stages you want to clear.
#   2. Set CONFIRM_RESET = True.
#   3. Run this cell.
#   4. Re-run from §1.4 (or the affected stage) to recompute.
#
# Available stage keys: "raw_corpus", "group_scans", "matched_df",
#                       "subtopics", "llm_labels", "bursts"

CONFIRM_RESET = False               # ← must be True to actually delete
RESET_STAGES  = []                  # ← list of stage keys to clear
# Example: RESET_STAGES = ["subtopics", "bursts"]
# To reset EVERYTHING: RESET_STAGES = list(CKPT.keys())

if not CONFIRM_RESET:
    print("CONFIRM_RESET is False — no action taken.")
    print("To clear caches, edit RESET_STAGES and set CONFIRM_RESET = True.")
    print(f"\nCurrently cached: {[k for k, p in CKPT.items() if p.exists()]}")
elif not RESET_STAGES:
    print("RESET_STAGES is empty — nothing to clear.")
else:
    print(f"Clearing {len(RESET_STAGES)} stage(s):")
    for stage in RESET_STAGES:
        if stage not in CKPT:
            print(f"  ✗ unknown stage: {stage}"); continue
        p = CKPT[stage]
        if p.is_file() and p.exists():
            p.unlink()
            print(f"  · removed {p.name}")
        elif p.is_dir() and p.exists():
            n = 0
            for f in p.iterdir():
                if f.is_file(): f.unlink(); n += 1
            print(f"  · cleared {p.name}/ ({n} file(s))")
        else:
            print(f"  · {stage}: nothing to clear")
    print("\n✓ Reset complete. Re-run §1.4 to confirm, then re-run from the top.")


---

## 10. What's *not* in this notebook (by design)

Trimmed during consolidation to keep the workflow tight:

| From | Section | Why omitted |
|---|---|---|
| NB1 | ETO seed-term TF-IDF | Backbone comes from inline scaffold; ETO is a §5.1 pluggable. |
| NB1 | UCSD 13-discipline scaffold + layer merging | Replaced by inline misinfo-focused backbone. |
| NB2 | Auto-extracted candidate keywords + author-keyword harvest | Superseded by `_GROUP_REGISTRY`. |
| NB2 | Single LDA over the whole corpus | Replaced by per-boolean-partition LDA (§4.2). |
| NB2 | `group_*` tagging + `group_mapping_df` | Boolean searches subsume this; `grp_*` columns feed boolean logic. |
| NB3 | Overlay projection | Different analytical move. Run `3-MediaMisinfo_backbone_overlay.ipynb` separately. |

## Tips for first-time runs

1. **First session**: open the notebook, click *Cell → Run All*. Don't supervise — go do something else for 90 minutes.
2. **When you come back**: if the kernel is still alive and §6 is showing burst heatmaps, you're done. Run §8.1 + §9.1 to finalize.
3. **If the kernel died** (timeout, OOM, network hiccup): start a fresh DataX session, open the notebook, *Run All* again. Cached steps will skip. You'll see exactly which stage continues.
4. **If something looks wrong** in a particular stage (e.g. you edited `_GROUP_REGISTRY` and want fresh scans): visit §9.2, set `RESET_STAGES = ["group_scans"]`, `CONFIRM_RESET = True`, run, then *Run All* from the top.
5. **Iterating on `BOOLEAN_SEARCHES`** is cheap — the §3.2a group scans don't need to re-run, only the §3.2b reduction. Set `RESET_STAGES = ["matched_df", "subtopics", "bursts"]` after editing.
